# Batch gradient descent for multiple linear regression

Diabetes data, 10 inputs.

In [ ]:
import time
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [ ]:
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
reg = LinearRegression().fit(X_train, y_train)
print("OLS test R2", r2_score(y_test, reg.predict(X_test)))
print(reg.coef_.round(1), round(reg.intercept_, 2))

## Our own batch gradient descent

In [ ]:
class GDRegressor:
    def __init__(self, learning_rate=0.01, epochs=100):
        self.lr, self.epochs = learning_rate, epochs

    def fit(self, X, y):
        n = X.shape[0]
        self.intercept_ = 0.0
        self.coef_ = np.ones(X.shape[1])           # start: all coefficients 1
        for _ in range(self.epochs):
            y_hat = X @ self.coef_ + self.intercept_    # predictions for ALL rows
            err = y - y_hat
            intercept_der = -2 * err.mean()           # dL/d(beta_0)
            coef_der = -2 * (X.T @ err) / n           # all m derivatives at once
            self.intercept_ -= self.lr * intercept_der
            self.coef_ -= self.lr * coef_der
        return self

    def predict(self, X):
        return X @ self.coef_ + self.intercept_

start = time.perf_counter()
gdr = GDRegressor(learning_rate=0.5, epochs=1000).fit(X_train, y_train)
print("seconds", round(time.perf_counter() - start, 3))
print("GD test R2", r2_score(y_test, gdr.predict(X_test)))
print(gdr.coef_.round(1), round(gdr.intercept_, 2))

## More epochs: gradient descent moves towards the OLS coefficients

In [ ]:
for epochs in [100, 1000, 10000, 50000]:
    g = GDRegressor(learning_rate=0.5, epochs=epochs).fit(X_train, y_train)
    print(epochs, "train R2", round(r2_score(y_train, g.predict(X_train)), 4),
          "test R2", round(r2_score(y_test, g.predict(X_test)), 4),
          "largest gap to OLS coef", round(np.abs(g.coef_ - reg.coef_).max(), 1))

## Why vectorise: the same derivative with a Python loop

In [ ]:
def coef_der_loop(X, err):
    n, m = X.shape
    der = np.zeros(m)
    for j in range(m):              # every coefficient
        total = 0.0
        for i in range(n):          # every row
            total += err[i] * X[i, j]
        der[j] = -2 * total / n
    return der

err = y_train - (X_train @ np.ones(10))
t = time.perf_counter(); d1 = coef_der_loop(X_train, err); t_loop = time.perf_counter() - t
t = time.perf_counter(); d2 = -2 * X_train.T @ err / len(err); t_vec = time.perf_counter() - t
print(np.allclose(d1, d2), f"loop {t_loop*1000:.2f} ms, vectorised {t_vec*1000:.3f} ms")